In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

PROJECT_ROOT = Path.cwd().parent
if PROJECT_ROOT.name != "cs3244_group12_imdb_spoilers":
    PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parents[1]

INPUT_PATH = PROJECT_ROOT / "data" / "cleaned" / "IMDB_reviews.json"
OUTPUT_DIRECTORY = PROJECT_ROOT / "data" / "splits"
TRAIN_PATH = OUTPUT_DIRECTORY / "IMDB_reviews_train.json"
TEST_PATH = OUTPUT_DIRECTORY / "IMDB_reviews_test.json"

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH} does not exist. Run `python src/data_cleaning.py` first."
    )

# Stratified train-test split

This split keeps every exact `review_text` value in one partition, as recommended in `docs/eda-takeaways.md`. Groups with conflicting `is_spoiler` labels are also kept intact, but are excluded from the stratification labels because no single group label can represent them without arbitrarily changing the data.

This is a review-text-grouped split, not the movie-disjoint split required for RQ3. The validation cell reports movie overlap so that the distinction is explicit.

In [ ]:
reviews = pd.read_json(INPUT_PATH, lines=True)
required_columns = {"review_text", "is_spoiler", "movie_id"}
missing_columns = required_columns.difference(reviews.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if reviews["review_text"].isna().any():
    raise ValueError("review_text contains missing values; cannot form leakage-safe groups.")

# One row per exact review-text group. Each group must stay in one split.
group_summary = (
    reviews.groupby("review_text", sort=False, dropna=False)
    .agg(
        group_size=("review_text", "size"),
        unique_labels=("is_spoiler", "nunique"),
    )
    .reset_index()
)
group_summary["group_label"] = (
    reviews.groupby("review_text", sort=False, dropna=False)["is_spoiler"]
    .first()
    .to_numpy()
)

stratifiable_groups = group_summary[group_summary["unique_labels"] == 1].copy()
conflicting_groups = group_summary[group_summary["unique_labels"] > 1].copy()

train_group_text, test_group_text = train_test_split(
    stratifiable_groups["review_text"],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratifiable_groups["group_label"],
)

# Assign conflicting-label groups as indivisible groups to the smaller partition.
train_group_text = list(train_group_text)
test_group_text = list(test_group_text)
group_sizes = stratifiable_groups.set_index("review_text")["group_size"]
train_rows = int(group_sizes.loc[train_group_text].sum())
test_rows = int(group_sizes.loc[test_group_text].sum())
conflicting_groups = conflicting_groups.sample(frac=1, random_state=RANDOM_STATE)

for _, group in conflicting_groups.iterrows():
    if train_rows <= test_rows:
        train_group_text.append(group["review_text"])
        train_rows += int(group["group_size"])
    else:
        test_group_text.append(group["review_text"])
        test_rows += int(group["group_size"])

train_text = set(train_group_text)
test_text = set(test_group_text)
train_reviews = reviews[reviews["review_text"].isin(train_text)].copy()
test_reviews = reviews[reviews["review_text"].isin(test_text)].copy()

if set(train_reviews.index).intersection(test_reviews.index):
    raise AssertionError("A row appears in both train and test partitions.")
if set(train_reviews["review_text"]).intersection(test_reviews["review_text"]):
    raise AssertionError("An exact review_text value appears in both partitions.")

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
train_reviews.to_json(TRAIN_PATH, orient="records", lines=True, force_ascii=False)
test_reviews.to_json(TEST_PATH, orient="records", lines=True, force_ascii=False)

print(f"Total reviews: {len(reviews):,}")
print(f"Training reviews: {len(train_reviews):,} ({len(train_reviews) / len(reviews):.2%})")
print(f"Test reviews: {len(test_reviews):,} ({len(test_reviews) / len(reviews):.2%})")
print(f"Conflicting-label text groups kept intact: {len(conflicting_groups):,}")
print(f"Saved training data to: {TRAIN_PATH}")
print(f"Saved test data to: {TEST_PATH}")

Total reviews: 573,913
Training reviews: 458,965 (79.97%)
Test reviews: 114,948 (20.03%)
Conflicting-label text groups kept intact: 78
Saved training data to: c:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\data\splits\IMDB_reviews_train.json
Saved test data to: c:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\data\splits\IMDB_reviews_test.json


In [3]:
def label_distribution(frame):
    """Return counts and percentages for the spoiler label."""
    return pd.DataFrame(
        {
            "count": frame["is_spoiler"].value_counts().sort_index(),
            "percentage": frame["is_spoiler"].value_counts(normalize=True).sort_index().mul(100).round(2),
        }
    )


print("Overall label distribution")
display(label_distribution(reviews))
print("Training label distribution")
display(label_distribution(train_reviews))
print("Test label distribution")
display(label_distribution(test_reviews))

train_movies = set(train_reviews["movie_id"].dropna())
test_movies = set(test_reviews["movie_id"].dropna())
movie_overlap = train_movies.intersection(test_movies)
print(f"Movies represented in both partitions: {len(movie_overlap):,}")
print("This is expected for this review-text-grouped split; use a separate movie-disjoint split for RQ3.")

Overall label distribution


,count,percentage
is_spoiler,,
False,422989,73.7
True,150924,26.3


Training label distribution


,count,percentage
is_spoiler,,
False,338295,73.71
True,120670,26.29


Test label distribution


,count,percentage
is_spoiler,,
False,84694,73.68
True,30254,26.32


Movies represented in both partitions: 1,569
This is expected for this review-text-grouped split; use a separate movie-disjoint split for RQ3.
